In [ ]:
import torch, random, numpy as np, os

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Repro-ish
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True  # speed


CUDA available: False


In [ ]:
# --- Clean mount + CSV checks (Colab safe) ---

import os, shutil, subprocess, sys
from pathlib import Path

MOUNT_POINT = "/content/drive"

# 1) Unmount if something is already mounted (ignore errors)
try:
    subprocess.run(["fusermount", "-u", MOUNT_POINT], check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
except Exception:
    pass  # ok if not mounted

# 2) Ensure mountpoint is empty (Colab refuses to mount over non-empty dirs)
if os.path.exists(MOUNT_POINT):
    # If directory exists and has contents, remove it completely
    try:
        if os.listdir(MOUNT_POINT):
            shutil.rmtree(MOUNT_POINT)
    except Exception:
        # If rmtree fails (rare), try a shell fallback
        os.system(f"rm -rf {MOUNT_POINT}")

# Re-create the (empty) mountpoint directory
os.makedirs(MOUNT_POINT, exist_ok=True)

# 3) Mount Google Drive cleanly
from google.colab import drive
drive.mount(MOUNT_POINT)  # you'll see the auth prompt here

# 4) Paths
ROOT = Path(MOUNT_POINT) / "MyDrive"
OUT_DIR = ROOT / "histo_outputs"  # this is where your CSVs & weights live
TRAIN_CSV = OUT_DIR / "train.csv"
VAL_CSV   = OUT_DIR / "val.csv"
TEST_CSV  = OUT_DIR / "test.csv"

# 5) Verify CSVs exist; show helpful diagnostics if any are missing
missing = [p for p in (TRAIN_CSV, VAL_CSV, TEST_CSV) if not p.exists()]
if not missing:
    print("✅ OK. CSVs found in:", OUT_DIR)
    for p in (TRAIN_CSV, VAL_CSV, TEST_CSV):
        print(" -", p)
else:
    print("❌ Missing file(s):")
    for p in missing:
        print(" -", p)
    print("\n🔎 Debug info:")
    if not OUT_DIR.exists():
        print(f" - Output folder not found: {OUT_DIR}")
        # List top-level MyDrive so you can verify where histo_outputs is
        top = sorted([str(x) for x in ROOT.glob("*")])
        print(" - Contents of MyDrive:")
        for x in top[:100]:
            print("   •", x)
    else:
        # List what's inside OUT_DIR to help you spot naming typos
        contents = sorted([str(x) for x in OUT_DIR.glob("*")])
        if contents:
            print(f" - Contents of {OUT_DIR}:")
            for x in contents[:200]:
                print("   •", x)
        else:
            print(f" - {OUT_DIR} exists but is empty.")
    # Hard-stop so your later code doesn't proceed with missing inputs
    sys.exit(1)


Mounted at /content/drive
✅ OK. CSVs found in: /content/drive/MyDrive/histo_outputs
 - /content/drive/MyDrive/histo_outputs/train.csv
 - /content/drive/MyDrive/histo_outputs/val.csv
 - /content/drive/MyDrive/histo_outputs/test.csv


In [ ]:
!pip -q install timm==1.0.9 torchmetrics==1.4.0 pandas==2.2.2 scikit-learn==1.5.1 --upgrade


In [ ]:
# === ONE-CELL: Force-include DATASET2 and rebuild 4-dataset splits ===
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# existing dataset roots
BREAKHIS_DIR = Path("/content/drive/MyDrive/data/BreaKHis_v1/histology_slides/breast")
LC25000_DIR  = Path("/content/drive/MyDrive/histo/LC25000")
MHIST_DIR    = Path("/content/drive/MyDrive/data3/MHIST")
DATA2_DIR    = Path("/content/drive/MyDrive/data2/archive")   # <— your dataset 2
SAVE_DIR     = Path("/content/drive/MyDrive/histo_outputs"); SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
def is_img(p: Path): return p.is_file() and p.suffix.lower() in IMG_EXTS
def guess_pid(name: str):
    m = re.search(r'(\d{5,})', name)
    if m: return m.group(1)
    return re.split(r'[_\-\s]+', Path(name).stem)[0]

rows=[]

# ---- BreakHis
if BREAKHIS_DIR.exists():
    for ct in ["benign","malignant"]:
        d = BREAKHIS_DIR/ct/"SOB"
        if not d.exists(): continue
        for subtype in d.iterdir():
            if not subtype.is_dir(): continue
            for pat in subtype.iterdir():
                if not pat.is_dir(): continue
                for mag in pat.iterdir():
                    if not mag.is_dir(): continue
                    for im in mag.iterdir():
                        if is_img(im):
                            rows.append(dict(filepath=str(im),dataset="BreakHis",
                                             label_raw=ct,label=1 if ct=="malignant" else 0,
                                             patient_id=pat.name))

# ---- LC25000
if LC25000_DIR.exists():
    for cls_dir in LC25000_DIR.iterdir():
        if not cls_dir.is_dir(): continue
        cname = cls_dir.name.lower()
        if cname.endswith("_n"): y=0
        elif cname.endswith("_aca") or cname.endswith("_scc"): y=1
        else: continue
        for im in cls_dir.iterdir():
            if is_img(im):
                rows.append(dict(filepath=str(im),dataset="LC25000",
                                 label_raw=cname,label=y,patient_id=guess_pid(im.name)))

# ---- MHIST
if MHIST_DIR.exists():
    ann = MHIST_DIR/"annotations.csv"; imgs = MHIST_DIR/"images"
    if ann.exists() and imgs.exists():
        dfm = pd.read_csv(ann)
        name_col = next((c for c in dfm.columns if "name" in c.lower() or c.lower()=="image"), dfm.columns[0])
        lab_col  = next((c for c in dfm.columns if "label" in c.lower()), dfm.columns[-1])
        for _,r in dfm.iterrows():
            fn = str(r[name_col]); fp = imgs/fn
            if not fp.exists():
                base = Path(fn).stem
                for ext in [".png",".jpg",".jpeg"]:
                    if (imgs/(base+ext)).exists(): fp = imgs/(base+ext); break
            if fp.exists() and is_img(fp):
                raw = str(r[lab_col]).strip()
                y = 1 if raw.upper()=="SSA" else 0
                rows.append(dict(filepath=str(fp),dataset="MHIST",
                                 label_raw=raw,label=y,patient_id=guess_pid(fp.name)))

# ---- DATASET2  (robust handling)
def try_read_csv(p: Path):
    if not p.exists(): return None
    try: return pd.read_csv(p)
    except: return None

def add_from_csv(df, img_root: Path):
    if df is None or df.empty: return 0
    # detect column names
    id_candidates = ["image","image_id","id","filename","file_name","image_name"]
    y_candidates  = ["label","target","class"]
    id_col = next((c for c in df.columns if c.lower() in id_candidates), None)
    y_col  = next((c for c in df.columns if c.lower() in y_candidates), None)
    if id_col is None or y_col is None: return 0
    added=0
    for _,r in df.iterrows():
        fn = str(r[id_col]); fp = img_root/fn
        if not fp.exists():
            base = Path(fn).stem
            if not Path(fn).suffix:
                for ext in [".png",".jpg",".jpeg",".tif",".tiff"]:
                    if (img_root/(base+ext)).exists(): fp = img_root/(base+ext); break
            if not fp.exists():
                hits = list(DATA2_DIR.rglob(Path(fn).name))
                if hits: fp = hits[0]
        if fp.exists() and is_img(fp):
            raw = str(r[y_col]).strip()
            try:
                y = int(float(raw)); y = 1 if y>=1 else 0
            except:
                y = 1 if raw.lower() in {"malignant","cancer","tumor","tumour","aca","scc","ssa","tum","tumorous"} else 0
            rows.append(dict(filepath=str(fp),dataset="DATASET2",
                             label_raw=raw,label=y,patient_id=guess_pid(fp.name)))
            added+=1
    return added

if DATA2_DIR.exists():
    n1 = add_from_csv(try_read_csv(DATA2_DIR/"train_data.csv"), DATA2_DIR/"train")
    n2 = add_from_csv(try_read_csv(DATA2_DIR/"valid_data.csv"), DATA2_DIR/"valid")
    # fallback: no csv or partial — infer from folder names OR filenames (e.g., TUM_* -> malignant)
    for split in ["train","valid","val","test"]:
        sd = DATA2_DIR/split
        if not sd.exists(): continue
        subs = [p for p in sd.iterdir() if p.is_dir()]
        if subs:
            for cls in subs:
                cname=cls.name.lower()
                y = 1 if any(k in cname for k in ["malig","cancer","tum","aca","scc","ssa"]) else 0
                for im in cls.rglob("*"):
                    if is_img(im):
                        rows.append(dict(filepath=str(im),dataset="DATASET2",
                                         label_raw=cname,label=y,patient_id=guess_pid(im.name)))
        else:
            # flat folder: infer from filename prefix ('TUM_' treated as malignant)
            for im in sd.glob("*"):
                if is_img(im):
                    y = 1 if im.name.upper().startswith("TUM") else 0
                    rows.append(dict(filepath=str(im),dataset="DATASET2",
                                     label_raw=im.name,label=y,patient_id=guess_pid(im.name)))

# ---- Build manifest
manifest = pd.DataFrame(rows).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
print("Collected images:", len(manifest))
assert not manifest.empty

# ---- per-dataset robust split (patient-level when possible)
def split_df(df_ds, seed=42):
    df_ds=df_ds.copy()
    ug = df_ds["patient_id"].astype(str).nunique()
    name = df_ds["dataset"].iloc[0]
    if ug>=3:
        gss1=GroupShuffleSplit(test_size=0.15, n_splits=1, random_state=seed)
        i_trv,i_te = next(gss1.split(df_ds, groups=df_ds["patient_id"].astype(str)))
        trv,te = df_ds.iloc[i_trv].copy(), df_ds.iloc[i_te].copy()
        gss2=GroupShuffleSplit(test_size=0.1765, n_splits=1, random_state=seed+1)
        i_tr,i_va = next(gss2.split(trv, groups=trv["patient_id"].astype(str)))
        tr,va = trv.iloc[i_tr].copy(), trv.iloc[i_va].copy()
        tr["split"]= "train"; va["split"]="val"; te["split"]="test"
        print(f"• {name}: groups={ug} (patient-level)")
        return pd.concat([tr,va,te])
    elif ug==2:
        g = df_ds["patient_id"].astype(str).unique()
        te = df_ds[df_ds["patient_id"].astype(str)==g[0]].copy(); te["split"]="test"
        trv= df_ds[df_ds["patient_id"].astype(str)!=g[0]].copy()
        tr,va = train_test_split(trv, test_size=0.1765/(1-0.15), random_state=seed, stratify=trv["label"])
        tr["split"]="train"; va["split"]="val"
        print(f"• {name}: groups=2 (mixed: patient+image-level)")
        return pd.concat([tr,va,te])
    else:
        tr,temp = train_test_split(df_ds, test_size=0.30, random_state=seed, stratify=df_ds["label"])
        va,te  = train_test_split(temp,  test_size=0.5, random_state=seed+1, stratify=temp["label"])
        tr["split"]="train"; va["split"]="val"; te["split"]="test"
        print(f"• {name}: groups=1 (image-level)")
        return pd.concat([tr,va,te])

parts=[]
for ds, df_ds in manifest.groupby("dataset", as_index=False):
    parts.append(split_df(df_ds))
manifest2 = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=123).reset_index(drop=True)

# save
manifest2.to_csv(SAVE_DIR/"manifest.csv", index=False)
for s in ["train","val","test"]:
    part = manifest2[manifest2["split"]==s]
    part.to_csv(SAVE_DIR/f"{s}.csv", index=False)
    print(f"{s}: {len(part)} | datasets ->", part["dataset"].value_counts().to_dict())

print("\nSaved to:", SAVE_DIR)


Mounted at /content/drive
Collected images: 45961
• BreakHis: groups=82 (patient-level)
• DATASET2: groups=9 (patient-level)
• LC25000: groups=25000 (patient-level)
• MHIST: groups=1 (image-level)
train: 30653 | datasets -> {'LC25000': 17499, 'DATASET2': 5500, 'BreakHis': 5448, 'MHIST': 2206}
val: 7591 | datasets -> {'LC25000': 3751, 'DATASET2': 2200, 'BreakHis': 1167, 'MHIST': 473}
test: 7717 | datasets -> {'LC25000': 3750, 'DATASET2': 2200, 'BreakHis': 1294, 'MHIST': 473}

Saved to: /content/drive/MyDrive/histo_outputs


In [ ]:
# ===== FINAL TRAINING: ViT-B/16 + DANN with early stop, TTA, temperature calibration, rich metrics =====
!pip -q install timm==1.0.9 torchmetrics==1.4.0 pandas==2.2.2 scikit-learn==1.5.1 --upgrade

import os, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm
from torch.amp import autocast, GradScaler
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report, accuracy_score, roc_curve
from tqdm.auto import tqdm

# ---------- config ----------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path("/content/drive/MyDrive/histo_outputs"); SAVE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = SAVE_DIR/"train.csv"; VAL_CSV = SAVE_DIR/"val.csv"; TEST_CSV = SAVE_DIR/"test.csv"
BEST_PATH = SAVE_DIR/"vitb16_dann_best.pth"

IMG_SIZE    = 224
BATCH_SIZE  = 64      # drop to 48/32 if OOM
NUM_WORKERS = 2
EPOCHS      = 30
LR_ENCODER  = 3e-5
LR_HEADS    = 3e-4
WD          = 5e-2
LABEL_SMOOTH = 0.05
LAMBDA_START = 0.10   # ramp from 0.10 -> 0.50
LAMBDA_END   = 0.50
PATIENCE     = 5      # early stopping on val acc
TTA_N        = 8      # 8x TTA at test
SEED         = 42
# ----------------------------

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --- utils ---
IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
def is_img(p): return Path(p).suffix.lower() in IMG_EXTS

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.4,0.4,0.2,0.05),
    transforms.GaussianBlur(kernel_size=3),
    transforms.RandomAutocontrast(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE+32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# simple TTA: flips + 90° rotations
tta_transforms = [
    transforms.Compose([transforms.Resize(IMG_SIZE+32), transforms.CenterCrop(IMG_SIZE), transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize(IMG_SIZE+32), transforms.CenterCrop(IMG_SIZE), transforms.RandomVerticalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]
def rotate_tensor(x, k):
    # rotate batch x by k*90 degrees
    return torch.rot90(x, k, dims=[-2,-1])

class HistoDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda x: isinstance(x,str) and is_img(x) and Path(x).exists())].copy()
        assert {"filepath","label","dataset"}.issubset(df.columns)
        self.df = df.reset_index(drop=True); self.tfm = tfm
        self.ds2id = {ds:i for i,ds in enumerate(sorted(df["dataset"].unique()))}
        self.df["ds_id"] = self.df["dataset"].map(self.ds2id)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r.filepath).convert("RGB")
        x = self.tfm(img)
        y = torch.tensor(int(r.label), dtype=torch.long)
        d = torch.tensor(int(r.ds_id), dtype=torch.long)
        return x, y, d, str(r.dataset)

train_ds, val_ds, test_ds = HistoDS(TRAIN_CSV, train_tfms), HistoDS(VAL_CSV, eval_tfms), HistoDS(TEST_CSV, eval_tfms)
num_domains = len(train_ds.ds2id)

# domain-balanced sampler: inverse freq of (dataset,label)
combo = train_ds.df[["dataset","label"]].astype(str).agg("||".join, axis=1)
freq = combo.value_counts().to_dict()
weights = [1.0/freq[c] for c in combo]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# --- Gradient Reversal Layer (for DANN) ---
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd): ctx.lambd = lambd; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lambd * g, None
def grad_reverse(x, lambd): return GradReverse.apply(x, lambd)

# --- Model ---
class ViT_DANN(nn.Module):
    def __init__(self, num_domains):
        super().__init__()
        self.encoder = timm.create_model("vit_base_patch16_224.augreg_in21k", pretrained=True, num_classes=0)
        feat_dim = self.encoder.num_features
        self.cancer_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, 2))
        self.domain_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim,128), nn.ReLU(inplace=True), nn.Linear(128,num_domains))
    def forward(self, x, lambd=0.0):
        feats = self.encoder(x)
        logits_c = self.cancer_head(feats)
        logits_d = self.domain_head(grad_reverse(feats, lambd) if lambd>0 else feats)
        return logits_c, logits_d

model = ViT_DANN(num_domains=num_domains).to(DEVICE)

enc_params = list(model.encoder.parameters())
head_params = list(model.cancer_head.parameters()) + list(model.domain_head.parameters())
optimizer = torch.optim.AdamW([
    {"params": enc_params, "lr": LR_ENCODER},
    {"params": head_params, "lr": LR_HEADS},
], weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler("cuda")
ce_main = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
ce_dom  = nn.CrossEntropyLoss()

def evaluate(loader, model, tta=False, temp=1.0):
    model.eval()
    probs_all, y_all, ds_all = [], [], []
    with torch.no_grad():
        for x,y,_,dsname in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if tta:
                # base forward
                logits,_ = model(x, lambd=0.0)
                logits = logits / temp
                p = torch.softmax(logits, dim=1)[:,1]
                # flips
                for tfm in tta_transforms:
                    xb = []
                    for i in range(x.size(0)):
                        im = transforms.ToPILImage()(x[i].cpu()*torch.tensor([0.229,0.224,0.225]).view(3,1,1)+torch.tensor([0.485,0.456,0.406]).view(3,1,1))
                        xb.append(tfm(im))
                    xb = torch.stack(xb).to(DEVICE)
                    logits,_ = model(xb, lambd=0.0); logits = logits / temp
                    p += torch.softmax(logits, dim=1)[:,1]
                # 90° rotations (k=1,2,3)
                for k in [1,2,3]:
                    xr = rotate_tensor(x, k)
                    logits,_ = model(xr, lambd=0.0); logits = logits / temp
                    p += torch.softmax(logits, dim=1)[:,1]
                p = p / (1 + len(tta_transforms) + 3)
            else:
                logits,_ = model(x, lambd=0.0); logits = logits / temp
                p = torch.softmax(logits, dim=1)[:,1]
            probs_all.extend(p.detach().cpu().tolist())
            y_all.extend(y.detach().cpu().tolist())
            ds_all.extend(list(dsname))
    return np.array(probs_all), np.array(y_all), np.array(ds_all)

def acc_auc(probs, ys):
    acc = accuracy_score(ys, (probs>=0.5).astype(int))
    try: auc = roc_auc_score(ys, probs)
    except: auc = float("nan")
    return acc, auc

def ece_score(probs, ys, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    inds = np.digitize(probs, bins) - 1
    ece = 0.0; N = len(ys)
    for b in range(n_bins):
        mask = inds==b
        if mask.sum()==0: continue
        conf = probs[mask].mean()
        acc = (probs[mask]>=0.5).astype(int).mean()
        ece += (mask.sum()/N) * abs(acc - conf)
    return float(ece)

# temperature scaling on val
class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.t = nn.Parameter(torch.ones(1))
    def forward(self, logits): return logits / self.t

best_val_acc = -1.0; bad_epochs = 0
for epoch in range(1, EPOCHS+1):
    model.train()
    running_main = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False)
    # ramp lambda
    lambd = LAMBDA_START + (LAMBDA_END - LAMBDA_START) * ((epoch-1)/(EPOCHS-1))
    for x,y,d,_ in pbar:
        x,y,d = x.to(DEVICE), y.to(DEVICE), d.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=torch.cuda.is_available()):
            log_c, log_d = model(x, lambd=lambd)
            loss_main = ce_main(log_c, y)
            loss_dom  = ce_dom(log_d, d)
            loss = loss_main + lambd * loss_dom
        if torch.cuda.is_available():
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        running_main += loss_main.item() * y.size(0)
        pbar.set_postfix(loss=f"{running_main/len(train_ds):.4f}", lambd=f"{lambd:.2f}")
    scheduler.step()

    # val (no TTA, temp=1)
    val_probs, val_y, _ = evaluate(val_loader, model, tta=False, temp=1.0)
    val_acc, val_auc = acc_auc(val_probs, val_y)
    print(f"Epoch {epoch:02d} | train_main_loss {running_main/len(train_ds):.4f} | val_acc {val_acc:.4f} | val_auc {val_auc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc; bad_epochs = 0
        torch.save({"state_dict": model.state_dict(), "val_acc": best_val_acc, "ds2id": train_ds.ds2id}, BEST_PATH)
        print("  ↳ saved best:", BEST_PATH)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val_acc={best_val_acc:.4f}")
            break

# --- Load best & calibration on val ---
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"]); model.eval()

# collect raw logits on val to fit temperature
val_logits_all, val_y = [], []
with torch.no_grad():
    for x,y,_,_ in val_loader:
        x = x.to(DEVICE)
        logits,_ = model(x, lambd=0.0)
        val_logits_all.append(logits.detach().cpu()); val_y.extend(y.tolist())
val_logits_all = torch.cat(val_logits_all, 0); val_y = np.array(val_y)
temp = TempScaler().to(DEVICE)
opt = torch.optim.LBFGS(temp.parameters(), lr=0.1, max_iter=50)
ce = nn.CrossEntropyLoss()
def _closure():
    opt.zero_grad()
    loss = ce(temp(val_logits_all.to(DEVICE)), torch.tensor(val_y, dtype=torch.long, device=DEVICE))
    loss.backward(); return loss
opt.step(_closure)
T = float(temp.t.detach().cpu().item())
print("Calibrated temperature T =", T)

# --- Test with TTA + temperature ---
test_probs, test_y, test_ds = evaluate(test_loader, model, tta=True, temp=T)
test_acc, test_auc = acc_auc(test_probs, test_y)
ece = ece_score(test_probs, test_y, n_bins=15)

# sensitivity/specificity at Youden J
fpr, tpr, thr = roc_curve(test_y, test_probs)
j_idx = int(np.argmax(tpr - fpr))
thr_opt = thr[j_idx] if j_idx < len(thr) else 0.5
pred_opt = (test_probs >= thr_opt).astype(int)
tn, fp, fn, tp = confusion_matrix(test_y, pred_opt).ravel()
sens = tp / (tp + fn + 1e-9)
spec = tn / (tn + fp + 1e-9)

print(f"\nTEST (TTA+Temp) => acc={test_acc:.4f} | auc={test_auc:.4f} | ece={ece:.4f} | thr*={thr_opt:.3f} | sens={sens:.4f} | spec={spec:.4f}")

# per-dataset metrics
df_eval = pd.DataFrame({"ds":test_ds, "y":test_y, "p":test_probs})
per_ds = []
for ds_name, g in df_eval.groupby("ds"):
    acc_ds = accuracy_score(g["y"], (g["p"]>=thr_opt).astype(int))
    try: auc_ds = roc_auc_score(g["y"], g["p"])
    except: auc_ds = float("nan")
    per_ds.append((ds_name, acc_ds, auc_ds, len(g)))
per_ds = sorted(per_ds, key=lambda x: x[0])
print("\nPer-dataset (thr*=Youden):")
for ds_name, a, u, n in per_ds:
    print(f"  {ds_name:10s}  n={n:4d}  acc={a:.4f}  auc={u:.4f}")

# confusion matrix + report at thr*
cm = confusion_matrix(test_y, pred_opt)
print("\nConfusion matrix (thr*):")
print(cm)
print("\nClassification report (thr*):\n", classification_report(test_y, pred_opt, digits=4))

# save predictions + summary for paper
df_eval["pred@thr*"] = pred_opt
df_eval.to_csv(SAVE_DIR/"vitb16_dann_test_predictions.csv", index=False)
with open(SAVE_DIR/"vitb16_dann_summary.txt","w") as f:
    f.write(f"TEST acc={test_acc:.4f} auc={test_auc:.4f} ece={ece:.4f} thr*={thr_opt:.3f} sens={sens:.4f} spec={spec:.4f}\n")
    for ds_name,a,u,n in per_ds:
        f.write(f"{ds_name}: n={n} acc={a:.4f} auc={u:.4f}\n")
print("\nSaved:", SAVE_DIR/"vitb16_dann_test_predictions.csv")
print("Saved:", SAVE_DIR/"vitb16_dann_summary.txt")


CUDA available: False


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Epoch 01:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
# ===== IMPROVED LODO (ViT-B/16 + DANN): longer training, temp calibration, rich metrics, aggregates =====
!pip -q install timm==1.0.9 pandas==2.2.2 scikit-learn==1.5.1 --upgrade

import os, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, confusion_matrix
from tqdm.auto import tqdm

# -------- config --------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path("/content/drive/MyDrive/histo_outputs")
MANIFEST = SAVE_DIR/"manifest.csv"
assert MANIFEST.exists(), "manifest.csv missing"

IMG_SIZE=224
BATCH_SIZE=32         # raise to 48/64 if VRAM allows
NUM_WORKERS=2
EPOCHS=24             # longer training per review
LR_ENCODER=3e-5
LR_HEADS=3e-4
WD=5e-2
LABEL_SMOOTH=0.05
LAMBDA_MAX=0.5        # DANN upper bound; ramps from 0 -> 0.5
VAL_FRAC=0.10
SEED=42
DANN_ABLATION=False   # set True to run baseline (no DANN) for comparison
# ------------------------

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
def is_img(p): return Path(p).suffix.lower() in IMG_EXTS

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.4,0.4,0.2,0.05),
    transforms.GaussianBlur(kernel_size=3),
    transforms.RandomAutocontrast(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE+32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

df_all = pd.read_csv(MANIFEST)
df_all = df_all[df_all["filepath"].apply(lambda x: isinstance(x,str) and is_img(x) and Path(x).exists())].copy()
datasets = sorted(df_all["dataset"].unique())
print("Datasets detected:", datasets)

class HistoDS(Dataset):
    def __init__(self, df, tfm, ds2id):
        self.df=df.reset_index(drop=True); self.tfm=tfm; self.ds2id=ds2id
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        img=Image.open(r.filepath).convert("RGB")
        x=self.tfm(img)
        y=torch.tensor(int(r.label),dtype=torch.long)
        d=torch.tensor(int(self.ds2id.get(r.dataset, 0)),dtype=torch.long)  # test set may have unseen domain id (unused)
        return x,y,d,r.dataset

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd): ctx.lambd=lambd; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lambd*g, None
def grad_reverse(x, lambd): return GradReverse.apply(x, lambd)

class ViT_DANN(nn.Module):
    def __init__(self, num_domains):
        super().__init__()
        self.encoder = timm.create_model("vit_base_patch16_224.augreg_in21k", pretrained=True, num_classes=0)
        feat_dim = self.encoder.num_features
        self.cancer_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, 2))
        self.domain_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim,128), nn.ReLU(inplace=True), nn.Linear(128,num_domains))
    def forward(self,x,lambd=0.0):
        feats=self.encoder(x)
        logits_c=self.cancer_head(feats)
        logits_d=self.domain_head(grad_reverse(feats,lambd) if lambd>0 else feats)
        return logits_c, logits_d

class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.t = nn.Parameter(torch.ones(1))
    def forward(self, logits): return logits / self.t

def fit_temperature(val_loader, model):
    model.eval()
    logits_all, y_all = [], []
    with torch.no_grad():
        for x,y,_,_ in val_loader:
            x = x.to(DEVICE)
            logits,_ = model(x, lambd=0.0)
            logits_all.append(logits.detach().cpu()); y_all.extend(y.tolist())
    logits_all = torch.cat(logits_all, 0)
    y_all = torch.tensor(y_all, dtype=torch.long)
    temp = TempScaler().to(DEVICE)
    opt = torch.optim.LBFGS(temp.parameters(), lr=0.1, max_iter=50)
    ce = nn.CrossEntropyLoss()
    def _closure():
        opt.zero_grad()
        loss = ce(temp(logits_all.to(DEVICE)), y_all.to(DEVICE))
        loss.backward(); return loss
    opt.step(_closure)
    return float(temp.t.detach().cpu().item())

def eval_probs(loader, model, temp=1.0):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for x,y,_,_ in loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            logits,_ = model(x, lambd=0.0)
            logits = logits / temp
            p = torch.softmax(logits, dim=1)[:,1]
            probs.append(p.detach().cpu()); ys.append(y.detach().cpu())
    probs = torch.cat(probs).numpy(); ys = torch.cat(ys).numpy()
    return probs, ys

def youden_threshold(y_true, probs):
    fpr, tpr, thr = roc_curve(y_true, probs)
    j = tpr - fpr
    idx = int(np.argmax(j))
    return float(thr[idx]) if idx < len(thr) else 0.5

def ece_score(probs, ys, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    inds = np.digitize(probs, bins)-1
    ece=0.0; N=len(ys)
    for b in range(n_bins):
        mask = inds==b
        if mask.sum()==0: continue
        conf = probs[mask].mean()
        acc  = (probs[mask]>=0.5).astype(int).mean()
        ece += (mask.sum()/N)*abs(acc-conf)
    return float(ece)

def run_lodo(held_out):
    # split
    df_tr = df_all[df_all["dataset"]!=held_out].copy()
    df_te = df_all[df_all["dataset"]==held_out].copy()
    # val split from training mix
    df_tr = df_tr.sample(frac=1.0, random_state=SEED)
    n_val = max(1, int(len(df_tr)*VAL_FRAC))
    df_va, df_tr = df_tr.iloc[:n_val].copy(), df_tr.iloc[n_val:].copy()

    ds2id = {ds:i for i,ds in enumerate(sorted(df_tr["dataset"].unique()))}
    num_domains = len(ds2id)

    tr_ds = HistoDS(df_tr, train_tfms, ds2id)
    va_ds = HistoDS(df_va, eval_tfms,  ds2id)
    te_ds = HistoDS(df_te, eval_tfms,  {**ds2id, held_out: len(ds2id)})

    # sampler: domain-balanced on (dataset,label)
    combo = tr_ds.df[["dataset","label"]].astype(str).agg("||".join, axis=1)
    freq = combo.value_counts().to_dict()
    weights = [1.0/freq[c] for c in combo]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = ViT_DANN(num_domains=num_domains).to(DEVICE)
    enc_params = list(model.encoder.parameters())
    head_params = list(model.cancer_head.parameters()) + list(model.domain_head.parameters())
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEADS},
    ], weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler("cuda")
    ce_main = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    ce_dom  = nn.CrossEntropyLoss()

    best_va_acc = -1.0
    for ep in range(1, EPOCHS+1):
        model.train()
        pbar = tqdm(tr_loader, desc=f"[{held_out}] Ep {ep:02d}", leave=False)
        lambd = 0.0 if DANN_ABLATION else (LAMBDA_MAX * ep / EPOCHS)
        for x,y,d,_ in pbar:
            x,y,d = x.to(DEVICE), y.to(DEVICE), d.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=torch.cuda.is_available()):
                log_c, log_d = model(x, lambd=lambd)
                loss = ce_main(log_c,y) + (0.0 if DANN_ABLATION else lambd*ce_dom(log_d,d))
            if torch.cuda.is_available():
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
        scheduler.step()

        # quick val acc to track convergence
        v_probs, v_y = eval_probs(va_loader, model, temp=1.0)
        v_acc = accuracy_score(v_y, (v_probs>=0.5).astype(int))
        best_va_acc = max(best_va_acc, v_acc)

    # calibrate temperature on val
    T = fit_temperature(va_loader, model)
    # evaluate test with calibrated logits
    te_probs, te_y = eval_probs(te_loader, model, temp=T)
    te_acc  = accuracy_score(te_y, (te_probs>=0.5).astype(int))
    try: te_auc = roc_auc_score(te_y, te_probs)
    except: te_auc = float("nan")
    ece = ece_score(te_probs, te_y, n_bins=15)
    thr = youden_threshold(te_y, te_probs)
    pred_opt = (te_probs>=thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(te_y, pred_opt).ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)

    return {
        "held_out": held_out,
        "val_best_acc": best_va_acc,
        "test_acc": te_acc,
        "test_auc": te_auc,
        "test_ece": ece,
        "thr_youden": thr,
        "sensitivity": sens,
        "specificity": spec,
        "temp": T,
        "train_n": len(tr_ds), "val_n": len(va_ds), "test_n": len(te_ds)
    }

rows=[]
for ds in datasets:
    rows.append(run_lodo(ds))
df_res = pd.DataFrame(rows)

# aggregates
def mean_std(s):
    return f"{s.mean():.3f} ± {s.std():.3f}"
print("\n=== LODO summary (calibrated) ===")
print("ACC :", mean_std(df_res["test_acc"]))
print("AUC :", mean_std(df_res["test_auc"]))
print("ECE :", mean_std(df_res["test_ece"]))
print("Sens:", mean_std(df_res["sensitivity"]))
print("Spec:", mean_std(df_res["specificity"]))
worst = df_res.loc[df_res["test_auc"].idxmin(), "held_out"]
best  = df_res.loc[df_res["test_auc"].idxmax(), "held_out"]
print(f"Worst dataset (AUC): {worst} | Best: {best}")

out = SAVE_DIR/f"LODO_vitb16_dann_calibrated_{'noDANN' if DANN_ABLATION else 'DANN'}.csv"
df_res.to_csv(out, index=False)
print("\nLODO per-dataset results:\n", df_res)
print("\nSaved:", out)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 888.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/410M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


[BreakHis] Ep 01:   0%|          | 0/1071 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ===== SUPER-FAST CPU EVAL for vitb16_dann_best.pth =====
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# deps (quiet)
import sys, subprocess
def pipi(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
try:
    import timm  # noqa
except:
    pipi(["timm==1.0.9"])
try:
    import sklearn, pandas, tqdm  # noqa
except:
    pipi(["scikit-learn","pandas","tqdm"])

# imports
import os, shutil, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, roc_curve, classification_report

# config
torch.set_grad_enabled(False)
torch.set_num_threads(max(2, os.cpu_count()//2))
DEVICE   = "cpu"
SAVE_DIR = Path("/content/drive/MyDrive/histo_outputs")
CKPT     = SAVE_DIR/"vitb16_dann_best.pth"
TEST_CSV = SAVE_DIR/"test.csv"
assert CKPT.exists(), f"Missing {CKPT}"
assert TEST_CSV.exists(), f"Missing {TEST_CSV}"

# copy test images to local (IO speedup)
df = pd.read_csv(TEST_CSV)
local_dir = Path("/content/test_local"); local_dir.mkdir(exist_ok=True)
def cp(p):
    p = Path(p); dst = local_dir/p.name
    if not dst.exists():
        try: shutil.copy2(p, dst)
        except: return None
    return str(dst)
df["filepath"] = [cp(p) for p in tqdm(df["filepath"], desc="Copying test images")]
df = df.dropna(subset=["filepath"]).reset_index(drop=True)
local_csv = Path("/content/test_local.csv"); df.to_csv(local_csv, index=False)
print(f"Local test files: {len(df)}")

# dataset/loader
IMG_SIZE   = 224
BATCH_SIZE = 32      # bump to 48/64 if RAM is fine
eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE+32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class HistoTest(Dataset):
    def __init__(self, csv_path):
        d = pd.read_csv(csv_path)
        self.fp, self.y, self.ds = d["filepath"].tolist(), d["label"].tolist(), d["dataset"].tolist()
    def __len__(self): return len(self.fp)
    def __getitem__(self, i):
        x = eval_tfms(Image.open(self.fp[i]).convert("RGB"))
        return x, int(self.y[i]), str(self.ds[i])

test_ds = HistoTest(local_csv)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=False)

# model (encoder + cancer head only)
import timm
class ViT_Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model("vit_base_patch16_224.augreg_in21k", pretrained=False, num_classes=0)
        feat = self.encoder.num_features
        self.cancer_head = nn.Sequential(nn.LayerNorm(feat), nn.Linear(feat, 2))
    def forward(self, x):
        feats = self.encoder(x)
        return self.cancer_head(feats)

# load checkpoint state dict (from DANN model) into this head-only module
sd = torch.load(CKPT, map_location=DEVICE)["state_dict"]
# strip keys not needed (domain head, if present)
sd2 = {k.replace("module.",""): v for k,v in sd.items() if not k.startswith("domain_head.")}
model = ViT_Head().to(DEVICE)
missing, unexpected = model.load_state_dict(sd2, strict=False)
# dynamic INT8 quantization (CPU speed)
model = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
model.eval()

# evaluate
@torch.no_grad()
def run_eval(loader, model):
    probs, ys, dsn = [], [], []
    pbar = tqdm(loader, desc="Evaluating", dynamic_ncols=True)
    for x,y,ds in pbar:
        x = x.to(DEVICE)
        logits = model(x)
        p = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        probs.extend(p.tolist()); ys.extend(y.numpy().tolist()); dsn.extend(list(ds))
    return np.array(probs), np.array(ys), np.array(dsn)

probs, ys, dsnames = run_eval(test_loader, model)

# metrics + save
acc = accuracy_score(ys, (probs>=0.5).astype(int))
try: auc = roc_auc_score(ys, probs)
except: auc = float("nan")
fpr, tpr, thr = roc_curve(ys, probs)
j = int(np.argmax(tpr - fpr)); thr_opt = thr[j] if j < len(thr) else 0.5
pred = (probs >= thr_opt).astype(int)
tn, fp, fn, tp = confusion_matrix(ys, pred).ravel()
sens = tp/(tp+fn+1e-9); spec = tn/(tn+fp+1e-9)

df_eval = pd.DataFrame({"ds":dsnames,"y":ys,"p":probs,"pred@thr*":pred})
per_ds = []
for name, g in df_eval.groupby("ds"):
    a = accuracy_score(g["y"], (g["p"]>=thr_opt).astype(int))
    try: u = roc_auc_score(g["y"], g["p"])
    except: u = float("nan")
    per_ds.append((name, len(g), a, u))
per_ds.sort(key=lambda z: z[0])

csv_path = SAVE_DIR/"vitb16_dann_test_predictions.csv"
txt_path = SAVE_DIR/"vitb16_dann_summary.txt"
df_eval.to_csv(csv_path, index=False)
with open(txt_path, "w") as f:
    f.write(f"TEST acc={acc:.4f} auc={auc:.4f} thr*={thr_opt:.3f} sens={sens:.4f} spec={spec:.4f}\n")
    f.write("Per-dataset (thr*=Youden):\n")
    for name,n,a,u in per_ds:
        f.write(f"{name}: n={n} acc={a:.4f} auc={u:.4f}\n")
    f.write("\nClassification report (thr*):\n")
    f.write(classification_report(ys, pred, digits=4))

print(f"\nSaved:\n- {csv_path}\n- {txt_path}")
print(f"\nOverall  acc={acc:.4f} | auc={auc:.4f} | thr*={thr_opt:.3f} | sens={sens:.4f} | spec={spec:.4f}")


Mounted at /content/drive


Copying test images: 100%|██████████| 7717/7717 [1:06:14<00:00,  1.94it/s]


Local test files: 7717


Evaluating: 100%|██████████| 242/242 [1:06:03<00:00, 16.38s/it]
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Saved:
- /content/drive/MyDrive/histo_outputs/vitb16_dann_test_predictions.csv
- /content/drive/MyDrive/histo_outputs/vitb16_dann_summary.txt

Overall  acc=0.9458 | auc=0.9858 | thr*=0.239 | sens=0.9421 | spec=0.9486
